# PySpark 8: nested and semi-structured data

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-08-nested-and-semi-structured-data.ipynb)

An order API returns each order as JSON with a nested customer, a list of line items and a map of attributes.
Clickstream events carry free-form JSON payloads whose fields change with every app release. You can keep this
data nested, flatten it, or query it in place, and each choice has a cost.

This lesson builds and reads structs, arrays and maps, turns arrays into rows without losing any, works on
arrays in place with higher-order functions, parses JSON strings, queries schemaless data with `VARIANT`, and
flattens a nested schema automatically.

Each section explains how something works, where it matters and what to watch out for, then runs small examples. Read each output, then the **What to notice** note under it. Run the cells in order.

**Prerequisites:** [PySpark 3: types, functions and nulls](pyspark-03-types-functions-and-nulls.ipynb) and [PySpark 7: aggregations and windows](pyspark-07-aggregations-and-windows.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0.*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

The first cell starts a local Spark session. The second creates `orders` (one row per order; order 12 lands just after midnight UTC) and `order_items` (one row per order line; each order's lines add up to its amount): a slice of the orders platform that every
module shares, introduced in [PySpark 1](pyspark-01-how-spark-runs.ipynb#setup-and-sample-data).

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import functions as F
import json

orders = spark.createDataFrame([
    (1, 1, "paid", datetime(2026, 1, 5, 9, 15), D("120.00")),
    (2, 1, "shipped", datetime(2026, 1, 5, 10, 5), D("84.50")),
    (3, 1, "returned", datetime(2026, 1, 5, 11, 40), D("30.00")),
    (4, 2, "paid", datetime(2026, 1, 6, 15, 20), D("260.00")),
    (5, 2, "placed", datetime(2026, 1, 7, 8, 5), D("55.25")),
    (6, 3, "shipped", datetime(2026, 1, 7, 17, 45), D("149.00")),
    (7, 3, "paid", datetime(2026, 1, 9, 12, 10), D("310.00")),
    (8, 4, "returned", datetime(2026, 1, 10, 19, 30), D("89.90")),
    (9, 4, "shipped", datetime(2026, 1, 12, 8, 55), D("1180.00")),
    (10, 5, "paid", datetime(2026, 1, 12, 13, 0), D("74.00")),
    (11, 6, "placed", datetime(2026, 1, 13, 10, 25), D("210.00")),
    (12, 6, "shipped", datetime(2026, 1, 14, 0, 30), D("96.40")),  # just after midnight UTC
], "order_id int, customer_id int, status string, order_ts timestamp, amount decimal(12,2)")

order_items = spark.createDataFrame([
    (1, "SKU-4001", 2, D("12.50")), (1, "SKU-4007", 1, D("95.00")),
    (2, "SKU-4002", 1, D("34.50")), (2, "SKU-4009", 2, D("25.00")),
    (3, "SKU-4002", 1, D("30.00")),
    (4, "SKU-4003", 1, D("149.00")), (4, "SKU-4004", 2, D("55.50")),
    (5, "SKU-4005", 1, D("55.25")),
    (6, "SKU-4003", 1, D("149.00")),
    (7, "SKU-4006", 1, D("189.00")), (7, "SKU-4008", 1, D("121.00")),
    (8, "SKU-4010", 1, D("89.90")),
    (9, "SKU-4011", 20, D("59.00")),  # a bulk reorder
    (10, "SKU-4012", 2, D("37.00")),
    (11, "SKU-4013", 1, D("210.00")),
    (12, "SKU-4014", 1, D("41.40")), (12, "SKU-4005", 1, D("55.00")),
], "order_id int, sku string, qty int, unit_price decimal(8,2)")

## Complex types: struct, array and map

| type | holds | example |
| --- | --- | --- |
| `struct<...>` | a fixed set of named fields, like a nested row | `customer: struct<id:int, tier:string>` |
| `array<T>` | an ordered list of values of one type | `skus: array<string>` |
| `map<K, V>` | key-value pairs with keys of one type and values of another | `attributes: map<string, string>` |

Build them with `F.struct`, `F.array`, `F.create_map` or `F.map_from_arrays`, and aggregate rows into arrays
with `collect_list` or `collect_set`. Types nest freely: an array of structs is how a JSON list of objects
arrives.

**Where it matters**
- **One row per business entity:** an order with its lines as an array of structs, instead of a separate table.
- **Flexible attributes:** a map for tags or settings whose keys vary by row.

**Watch out for**
- Very large arrays in one row (millions of elements) are processed by one task and can exhaust its memory.
- Maps cannot be grouped or compared directly; extract the keys you need first.

### Example

Turn the flat `order_items` table into one row per order with an array of line structs, a total and a map of SKU to quantity.

In [ ]:
nested = (
    order_items
    .groupBy("order_id")
    .agg(
        F.collect_list(F.struct("sku", "qty", "unit_price")).alias("lines"),
        F.sum(F.col("qty") * F.col("unit_price")).alias("total"),
        F.map_from_entries(F.collect_list(F.struct("sku", "qty"))).alias("qty_by_sku"),
    )
    .join(orders.select("order_id", F.struct("customer_id", "status").alias("order_info")), "order_id")
)
nested.printSchema()
nested.where("order_id IN (1, 9)").orderBy("order_id").show(truncate=False)

**What to notice:** Each order is now one row. `lines` is an array of structs, one per order line; `qty_by_sku` is a map from SKU to
quantity; `order_info` is a struct holding two columns. The schema tree shows every nested field and its type.

## Work with structs

Read a struct field with a dotted path (`F.col("order_info.status")`) or expand every field with
`select("order_info.*")`. Change fields without rebuilding the struct using `withField(name, expr)` and
`dropFields(name)`. `F.to_json(struct)` serializes a struct to a JSON string.

**Where it matters**
- **Selecting a few fields** from a wide nested payload.
- **Producing nested output** for APIs and document stores.

**Watch out for**
- `select("s.*")` can produce duplicate column names if the struct shares names with top-level columns.
- Reading one field of a struct in Parquet reads only that field's column, so nested data is still prunable.

### Example

Read a struct field, expand a struct, change a field with `withField`, and serialize to JSON.

In [ ]:
nested.select("order_id", F.col("order_info.status").alias("status")).orderBy("order_id").show(3)
nested.select("order_id", "order_info.*").orderBy("order_id").show(3)
nested.select(
    "order_id",
    F.col("order_info").withField("status", F.upper("order_info.status")).dropFields("customer_id").alias("info"),
    F.to_json("order_info").alias("info_json"),
).orderBy("order_id").show(3, truncate=False)

**What to notice:** The dotted path pulled out one field; `order_info.*` turned both fields into top-level columns. `withField`
replaced `status` inside the struct and `dropFields` removed `customer_id`, without listing the other fields.
`to_json` produced a compact JSON string ready to send to another system.

## Work with arrays

Useful array functions: `size`, `element_at(arr, n)` (1-based; negative counts from the end), `array_contains`,
`array_position`, `slice`, `sort_array`, `array_distinct`, `array_union`, `array_intersect`, `array_except`,
`arrays_zip` (pair elements of several arrays), `flatten`, and `array_join` (to a string).

**Where it matters**
- **Tags and labels:** does this product have the tag `sale`?
- **Ordered lists:** first and last page of a session, top three items.

**Watch out for**
- `element_at` past the end returns null (or raises in ANSI mode when you use `arr[n]` indexing). `element_at` is the safe form.
- `size(null)` is null in ANSI mode, -1 otherwise; handle missing arrays explicitly.

### Example

Inspect each order's SKUs: count, first and last, membership, sorting, and joining into a string.

In [ ]:
skus = nested.select("order_id", F.transform("lines", lambda line: line["sku"]).alias("skus"))
skus.select(
    "order_id", "skus",
    F.size("skus").alias("n"),
    F.element_at("skus", 1).alias("first"),
    F.element_at("skus", -1).alias("last"),
    F.array_contains("skus", "SKU-4005").alias("has_4005"),
    F.array_join(F.sort_array("skus"), ", ").alias("sorted_list"),
).orderBy("order_id").show(6, truncate=False)

**What to notice:** `element_at(skus, -1)` read the last element, and `array_contains` tested membership without exploding the
array. The order of elements inside `collect_list` follows the order rows were processed in, which is not
guaranteed; `sort_array` gives a stable order when it matters.

## Turn arrays into rows without losing any

`explode(arr)` returns one row per element. `explode_outer` also keeps rows whose array is null or empty, with a
null element. `posexplode` and `posexplode_outer` add the element's position. `inline(array_of_structs)` explodes
an array of structs straight into columns. Exploding a map gives one row per key-value pair.

**Where it matters**
- **Normalizing nested data** into a fact table: one row per order line.
- **Counting elements** across rows: which tags are most common?

**Watch out for**
- `explode` silently drops rows whose array is null or empty: a classic correctness bug. Use `explode_outer` unless dropping them is intended.
- Exploding multiplies rows; every later shuffle and write handles more data.

### Example

Explode a small set of events where one has a null list and one an empty list, with each function.

In [ ]:
tagged = spark.createDataFrame(
    [("e1", ["sale", "new"]), ("e2", ["sale"]), ("e3", None), ("e4", [])], "event_id string, tags array<string>"
)
print("explode:      ", tagged.select("event_id", F.explode("tags")).count(), "rows")
print("explode_outer:", tagged.select("event_id", F.explode_outer("tags")).count(), "rows")
tagged.select("event_id", F.posexplode_outer("tags").alias("position", "tag")).show()

nested.where("order_id IN (1, 2)").select("order_id", F.inline("lines")).show()
nested.where("order_id = 1").select("order_id", F.explode("qty_by_sku").alias("sku", "qty")).show()

**What to notice:** `explode` returned 3 rows and lost `e3` and `e4` entirely; `explode_outer` returned 5, keeping both with a null
tag. `posexplode_outer` added each tag's position. `inline` turned the array of line structs directly into
`sku`, `qty` and `unit_price` columns, and exploding the map gave one row per key-value pair.

## Transform arrays in place with higher-order functions

Higher-order functions take a lambda and apply it to array elements without exploding: `transform` (map each
element), `filter` (keep some), `exists` and `forall` (test), `aggregate` (fold into one value), and `zip_with`
(combine two arrays element by element). They run inside the JVM, keep one row per input row, and avoid an
explode-then-group round trip with its shuffle.

**Where it matters**
- **Per-row calculations** over line items: order totals, number of discounted lines.
- **Cleaning lists:** trim and lowercase every tag.

**Watch out for**
- The lambda is a column expression, not Python: use `F` functions inside it.
- In `aggregate`, the start value and the lambda's result must have the same type. Decimal arithmetic widens the type at each step, so cast both (for example to `decimal(20,2)`).

### Example

Compute each order's line amounts, total, number of multi-unit lines and whether all lines cost under 200, all without exploding.

In [ ]:
from decimal import Decimal

nested.select(
    "order_id",
    F.transform("lines", lambda l: l["qty"] * l["unit_price"]).alias("line_amounts"),
    F.aggregate(
        "lines",
        F.lit(Decimal("0.00")).cast("decimal(20,2)"),  # the accumulator's type...
        lambda acc, l: (acc + l["qty"] * l["unit_price"]).cast("decimal(20,2)"),  # ...must not change at each step
    ).alias("total"),
    F.size(F.filter("lines", lambda l: l["qty"] > 1)).alias("multi_unit_lines"),
    F.forall("lines", lambda l: l["unit_price"] < 200).alias("all_under_200"),
    F.exists("lines", lambda l: l["sku"] == "SKU-4005").alias("has_4005"),
).orderBy("order_id").show(6, truncate=False)

**What to notice:** Every order kept exactly one row. `aggregate` computed the same totals as the `groupBy` earlier. Note the two
casts: the accumulator must keep one type for the whole fold, but decimal addition widens the type
(`decimal(20,2) + decimal(20,2)` is `decimal(21,2)`), so both the start value and each step are cast back to
`decimal(20,2)`. Without them, Spark refuses the query with a type mismatch. Order 1 has one multi-unit line (two of `SKU-4001`); order 4 has one line
priced 149.00 and one at 55.50, so all its prices are under 200.

## Work with maps

Read a value with `F.col("m")["key"]` or `F.element_at("m", "key")`; list keys and values with `map_keys` and
`map_values`; combine maps with `map_concat`; and transform them with `map_filter`, `transform_keys` and
`transform_values`.

**Where it matters**
- **Sparse attributes** such as product properties or HTTP headers.
- **Lookups held in a row:** a map of currency to rate.

**Watch out for**
- A missing key returns null. Check `map_contains_key` when absence and null mean different things.
- Duplicate keys raise an error by default when building a map.

### Example

Read map values, list keys, and keep only SKUs ordered more than once.

In [ ]:
nested.select(
    "order_id", "qty_by_sku",
    F.col("qty_by_sku")["SKU-4001"].alias("qty_4001"),
    F.map_keys("qty_by_sku").alias("skus"),
    F.map_filter("qty_by_sku", lambda k, v: v > 1).alias("multi_unit"),
).orderBy("order_id").show(5, truncate=False)

**What to notice:** `qty_by_sku["SKU-4001"]` returned 2 for order 1 and null for orders without that key. `map_filter` kept only the
entries whose quantity is above 1, which is empty for most orders.

## Parse and produce JSON strings

Many sources deliver JSON as a string column. `F.from_json(col, schema)` parses it into a struct with the schema
you give; fields missing from the JSON become null and extra fields are ignored. `F.schema_of_json(sample)`
suggests a schema from one example. `get_json_object(col, "$.path")` extracts one value as a string without a
schema, and `to_json` goes the other way.

**Where it matters**
- **Kafka messages and webhooks,** which arrive as JSON text.
- **Database columns of JSON** exported to the lake.

**Watch out for**
- Malformed JSON parses to null; count nulls after parsing.
- `get_json_object` re-parses the string on every call; parse once with `from_json` when you need several fields.

### Example

Parse JSON payloads with an explicit schema, ask Spark to suggest a schema, and extract one field without a schema.

In [ ]:
payloads = spark.createDataFrame(
    [("w1", '{"customer": "c1", "items": [{"sku": "A", "qty": 2}], "coupon": "WELCOME"}'),
     ("w2", '{"customer": "c2", "items": []}'),
     ("w3", "not json")],
    "event_id string, payload string",
)
schema = "customer string, items array<struct<sku:string, qty:int>>, coupon string"
parsed = payloads.withColumn("data", F.from_json("payload", schema))
parsed.select("event_id", "data.customer", F.size("data.items").alias("items"), "data.coupon").show()

print(spark.range(1).select(F.schema_of_json(F.lit(payloads.first().payload))).first()[0])
payloads.select("event_id", F.get_json_object("payload", "$.items[0].sku").alias("first_sku")).show()

**What to notice:** `w2` has no coupon, so the field is null. `w3` is not JSON, so every parsed field is null for it, which is why you
count nulls after parsing. `schema_of_json` turned one example into a DDL schema you can paste and adjust, and
`get_json_object` pulled one value out by path.

## Query schemaless data with VARIANT

Spark 4 adds the `VARIANT` type: JSON stored in a compact binary form that can be queried without declaring a
schema. `F.parse_json` converts a JSON string once; `F.variant_get(v, "$.path", "type")` extracts and casts a
value; `F.try_variant_get` returns null instead of failing when the cast does not fit; `F.schema_of_variant`
describes what is inside.

**Where it matters**
- **Bronze tables** for payloads whose fields change every release.
- **Exploration** before deciding which fields to promote to typed columns in silver.

**Watch out for**
- Promote stable, frequently used paths to typed columns; extracting from `VARIANT` on every query costs more than reading a column.
- Check that your table format and every engine that reads the table support `VARIANT`.

### Example

Store three payloads as `VARIANT`, one with a field nobody declared, and extract paths with and without failure on bad types.

In [ ]:
raw = spark.createDataFrame(
    [(1, '{"device": "ios", "cart": {"items": 3, "coupon": "WELCOME10"}}'),
     (2, '{"device": "web", "cart": {"items": "many"}}'),
     (3, '{"device": "android", "cart": {"items": 5}, "beta": true}')],
    "event_id int, payload string",
)
variants = raw.select("event_id", F.parse_json("payload").alias("v"))
variants.select(
    "event_id",
    F.variant_get("v", "$.device", "string").alias("device"),
    F.try_variant_get("v", "$.cart.items", "int").alias("items"),
    F.try_variant_get("v", "$.beta", "boolean").alias("beta"),
    F.schema_of_variant("v").alias("schema"),
).show(truncate=False)

**What to notice:** Event 2 sends `items` as the string `many`, so `try_variant_get` returned null instead of failing the query.
Event 3 carries a `beta` field that no schema declared, and it can be queried anyway. `schema_of_variant` shows
each row's actual structure, which differs from row to row.

## Flatten a nested schema

Downstream tools often want flat columns. A small recursive function can walk the schema and replace every struct
field with a top-level column named by its path (`order_info_status`). Arrays are left alone, because flattening
them changes the number of rows; explode them deliberately first.

**Where it matters**
- **Loading nested JSON into a warehouse** or a CSV export that needs flat columns.
- **Generic ingestion code** that must work for many sources.

**Watch out for**
- Very deep or wide structs produce hundreds of columns; flatten only what consumers use.
- Path-based names can collide (`a.b_c` and `a_b.c`); pick a separator that does not appear in field names.

### Example

Flatten every struct in a two-level nested row into top-level columns.

In [ ]:
from pyspark.sql import types as T


def flatten_structs(df, separator="_"):
    """Replace struct columns with one column per leaf field, repeatedly, until none are left."""
    while True:
        structs = [f for f in df.schema.fields if isinstance(f.dataType, T.StructType)]
        if not structs:
            return df
        columns = []
        for field in df.schema.fields:
            if isinstance(field.dataType, T.StructType):
                columns += [F.col(f"`{field.name}`.`{child.name}`").alias(f"{field.name}{separator}{child.name}")
                            for child in field.dataType.fields]
            else:
                columns.append(F.col(f"`{field.name}`"))
        df = df.select(columns)

deep = spark.createDataFrame(
    [(1, ("c1", ("gold", 2024)), ["SKU-4001"])],
    "order_id int, customer struct<id:string, loyalty:struct<tier:string, since:int>>, skus array<string>",
)
flat = flatten_structs(deep)
flat.printSchema()
flat.show()

**What to notice:** The two levels of structs became `customer_id`, `customer_loyalty_tier` and `customer_loyalty_since`. The `skus`
array was left as it is: flattening an array would change the number of rows, so that is a separate, deliberate
`explode`.

## Check yourself

Answer each question in your own words before reading the answer.

**Q1.** When would you keep order lines as an array of structs instead of a separate table?

**A.** When consumers usually read an order with all its lines, or the data arrives that way (an API or document store). A separate table is better when lines are queried and aggregated on their own.

**Q2.** What rows does `explode` drop, and what do you use instead?

**A.** Rows whose array is null or empty. `explode_outer` keeps them with a null element.

**Q3.** Why use `aggregate` or `transform` rather than explode and `groupBy`?

**A.** Higher-order functions work inside each row, so they need no shuffle and keep one row per input row.

**Q4.** What happens to malformed JSON in `from_json`?

**A.** Every field of the parsed struct is null for that row; count nulls after parsing to find them.

**Q5.** When is `VARIANT` the right choice?

**A.** For payloads whose fields change often, typically in bronze. Promote stable, frequently queried paths to typed columns in silver.

**Q6.** Why does a generic flattening function leave arrays alone?

**A.** Flattening a struct keeps the row count; flattening an array changes it, which must be a deliberate decision.

## Key takeaways

- Structs, arrays and maps model nested data directly; Parquet still reads only the fields you use.
- Use `explode_outer` unless losing empty or null arrays is intended.
- Prefer higher-order functions to explode-and-regroup for per-row calculations.
- Parse JSON once with an explicit schema, then count nulls.
- Land changing payloads as `VARIANT`; promote stable paths to columns.

**Next:** Practise this module in [PySpark drills 8: nested and semi-structured data](pyspark-drills-08-nested-and-semi-structured-data.ipynb), then continue with [PySpark 9: Python UDFs and pandas](pyspark-09-python-udfs-and-pandas.ipynb).